# graph

> CodeGraph: call-graph layer for kosha, backed by graph.db. Augments semantic search with structural traversal.

In [ ]:
#| default_exp graph

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import ast, re, os
from json import loads as jl
from collections import defaultdict
from litesearch import *
from fastcore.all import (Path, L, patch, parallel_async, tuplify, first, fdelegates, globtastic, bind, true, dict2obj,
                          listify, filter_keys, merge, in_)
from kosha.core import arun, Kosha, parse, has_init, env_pkg_versions
from pyan.analyzer import CallGraphVisitor
from pyan.anutils import Scope, ExecuteInInnerScope

In [ ]:
#| export
try:
	# Bug 1 fix: auto-register missing nested lambda scopes
	def _safe_enter(self):
		o = self.analyzer
		o.name_stack.append(self.scopename)
		try: inner_ns = o.get_node_of_current_namespace().get_name()
		except Exception: inner_ns = '.'.join(o.name_stack)
		if inner_ns not in o.scopes: o.scopes[inner_ns] = Scope.from_names(self.scopename, [])
		o.scope_stack.append(o.scopes[inner_ns])
		o.context_stack.append(self.scopename)
		self.inner_ns = inner_ns
		return self
	ExecuteInInnerScope.__enter__ = _safe_enter

	# Bug 2 fix: guard None namespace in postprocessing
	_ANON = {'<listcomp>','<setcomp>','<dictcomp>','<genexpr>','<lambda>'}
	_orig_gp = CallGraphVisitor.get_parent_node
	def _safe_gp(self, n):
		if n.namespace is None: return None
		return _orig_gp(self, n)
	def _safe_collapse(self):
		for name in list(self.nodes):
			if name.partition('.')[0] not in _ANON: continue
			for n in self.nodes[name]:
				if n.namespace is None: continue
				pn = self.get_parent_node(n)
				if pn is None: continue
				for n2 in self.uses_edges.get(n, []): self.add_uses_edge(pn, n2)
				n.defined = False
	CallGraphVisitor.get_parent_node = _safe_gp
	CallGraphVisitor.collapse_inner  = _safe_collapse
except ImportError: pass

In [ ]:
#| export
class CodeGraph:
	'''Call graph builder and query engine. Uses pyan3 for static analysis and custom AST parsing for dynamic edges.
	Stores everything in a SQLite database.
	'''
	def __init__(self, path: str | Path = ':memory:'):
		self.path = path
		self.db = database(path)
		ge, gn, cd, fi = self.db.t.graph_edges, self.db.t.graph_nodes, self.db.t.co_dispatch, self.db.t.file_index
		ge.create(id=int, caller=str, callee=str, kind=str, confidence=float, if_not_exists=True, pk='id')
		ge.create_index(['caller'], if_not_exists=True)
		ge.create_index(['callee'], if_not_exists=True)
		ge.create_index(['kind'], if_not_exists=True)
		gn.create(node=str, flavor=str, file=str, pagerank=float, in_degree=int,
				  out_degree=int, if_not_exists=True, pk='node')
		cd.create(group_id=int, node=str, if_not_exists=True)
		fi.create(path=str, root=str, last_analyzed_at=float, if_not_exists=True, pk='path')
		fi.create_index(['root'], if_not_exists=True)
		self.ge, self.gn, self.cd, self.fi = ge, gn, cd, fi

	def nuke(self): [t.drop() for t in (self.ge, self.gn, self.cd, self.fi)]

In [ ]:
self = CodeGraph(':memory:')
tables = {r[0] for r in self.db.execute("SELECT name FROM sqlite_master WHERE type='table'")}
assert {'graph_edges','graph_nodes','co_dispatch','file_index'} <= tables, f"Missing tables: {tables}"
print("schema ok:", tables)

schema ok: {'co_dispatch', 'graph_nodes', 'file_index', 'graph_edges'}


In [ ]:
#| export
_reg_re = re.compile(
    r'^(add|register|include|on|connect|mount|sub(scribe)?|listen|use|route|handle|hook|attach|bind|plug|get|post|'
    r'put|delete|patch|head|options|before|after|middleware)(_\w+)?$', re.IGNORECASE)
_conn = {'connect', 'setup', 'register', 'init_app', 'create_app', 'mount','configure','bootstrap','install','activate'}
_web_pkg = {'fasthtml', 'flask', 'fastapi', 'starlette', 'django', 'aiohttp','sanic','tornado','litestar','blacksheep'}

In [ ]:
#| export
def _root(n):
	while isinstance(n, ast.Attribute): n = n.value
	return n.id if isinstance(n, ast.Name) else None

def _conf(root, verb, imp):
	pkg = imp.get(root) or next((v for k,v in imp.items() if k.startswith('*')), None)
	return 0.95 if pkg in _web_pkg else 0.75 if _reg_re.match(verb or '') else 0.5

def _e(caller, callee, kind, conf): return {'caller':caller,'callee':callee,'kind':kind,'confidence':conf}

def _dec_edges(n, mod, imp):
	fn = f"{mod}.{n.name}"
	return [_e(f"{mod}.{r}", fn, 'decorator', _conf(r, inner.attr, imp))
	        for dec in n.decorator_list
	        for inner in [dec.func if isinstance(dec,ast.Call) else dec]
	        if isinstance(inner,ast.Attribute) and (r := _root(inner))]

def _conn_edges(n, mod):
	if n.name not in _conn or not n.args.args: return []
	param, fn = n.args.args[0].arg, f'{mod}.{n.name}'
	return [_e(fn, f'{mod}.{a.id}', 'connect_body', 0.9)
	        for child in ast.walk(n)
	        if isinstance(child,ast.Call) and isinstance(child.func,ast.Attribute)
	        and _root(child.func) == param
	        for a in child.args if isinstance(a,ast.Name)]

def _reg_edges(n, mod, imp, all_fns):
	if not (isinstance(n,ast.Call) and isinstance(n.func,ast.Attribute) and _reg_re.match(n.func.attr)): return []
	root = _root(n.func) or 'unknown'
	return [_e(f"{mod}.{root}", f"{mod}.{a.id}", 'registration', _conf(root, n.func.attr, imp))
	        for a in n.args if isinstance(a,ast.Name) and a.id in all_fns]

def _co_edges(n, mod, top_fns):
	if not (isinstance(n,ast.Assign) and isinstance(getattr(n,'parent',None), ast.Module)): return []
	val = n.value
	grp = ([v.id for v in val.values if isinstance(v,ast.Name) and v.id in top_fns] if isinstance(val,ast.Dict) else
	       [e.id for e in val.elts if isinstance(e,ast.Name) and e.id in top_fns]
	       if isinstance(val,(ast.List,ast.Tuple)) else [])
	return [_e(f"{mod}.{a}",f"{mod}.{b}",'co_dispatch',0.8) for i,a in enumerate(grp) for b in grp[i+1:]]

def _patch_edges(n, mod):
	'fastcore @patch: @patch def fn(self:Class) → Class gains fn as method.'
	if not any(('patch' in ast.unparse(d)) for d in n.decorator_list): return []
	if not ((args := n.args.args) and (ann:=args[0].annotation)): return []
	tgts = ([ast.unparse(ann.left), ast.unparse(ann.right)]
			   if isinstance(ann, ast.BinOp) and isinstance(ann.op, ast.BitOr)
			   else [ast.unparse(ann)])
	return [_e(f"{mod}.{t}", f"{mod}.{n.name}", 'patch', 0.9) for t in tgts]

def _delegates_edges(n, mod):
	'fastcore @delegates(target): fn wraps target\'s signature — structural dependency.'
	is_nm, is_call = lambda d: isinstance(d, ast.Name) and d.id == 'delegates', lambda d: isinstance(d, ast.Call)
	is_attr = lambda d: isinstance(d, ast.Attribute) and d.attr == 'delegates'
	is_del = lambda d: is_call(d) and (is_nm(d.func) or is_attr(d.func))
	for d in n.decorator_list:
		if not (is_del(d) and (v:=d.args[0] if d.args else first(d.keywords, lambda kw:kw.arg == 'to'))): continue
		return [_e(f"{mod}.{n.name}", f"{mod}.{ast.unparse(v)}", 'delegates', 0.85)]
	return []

def dyn_edges(src:str, module:str) -> list[dict]:
	"All dynamic dispatch edges: decorator, connect_body, registration, co_dispatch."
	tree, imp, top, all_fns, *_ = parse(src)
	if tree is None: return []
	fns = L(ast.walk(tree)).filter(lambda n: isinstance(n,(ast.FunctionDef,ast.AsyncFunctionDef)))
	return (sum(fns.map(lambda n: _dec_edges(n, module, imp)), []) +
	        sum(fns.map(lambda n: _conn_edges(n, module)), []) +
			sum(fns.map(lambda n: _patch_edges(n, module)), []) +
			sum(fns.map(lambda n: _delegates_edges(n, module)), []) +
	        sum(L(ast.walk(tree)).map(lambda n: _reg_edges(n, module, imp, all_fns)), []) +
	        sum(L(tree.body).filter(lambda n: isinstance(n,ast.Assign))
	            .map(lambda n: _co_edges(n, module, top)), []))

In [ ]:
_SRC = '''
import flask
app = flask.Flask(__name__)

@app.route('/hello')
def hello(): pass

def setup(other_app):
    other_app.register(hello)
'''
edges = dyn_edges(_SRC, 'mymod')
kinds = {e['kind'] for e in edges}
assert 'decorator' in kinds, f"Expected decorator edge, got: {edges}"
assert 'registration' in kinds, f"Expected registration edge, got: {edges}"
assert dyn_edges('def (broken', 'bad') == L()
print("_dyn_edges ok:", edges)

_dyn_edges ok: [{'caller': 'mymod.app', 'callee': 'mymod.hello', 'kind': 'decorator', 'confidence': 0.75}, {'caller': 'mymod.setup', 'callee': 'mymod.hello', 'kind': 'connect_body', 'confidence': 0.9}, {'caller': 'mymod.other_app', 'callee': 'mymod.hello', 'kind': 'registration', 'confidence': 0.75}]


In [ ]:
_DELEGATES_SRC = '''
from fastcore.all import delegates

def target(x, y, z=1): pass

@delegates(target)
def wrapper_pos(**kw): pass

@delegates(to=target)
def wrapper_kw(**kw): pass

@delegates
def bare(**kw): pass

@delegates()
def empty(**kw): pass
'''
dyn_edges(_DELEGATES_SRC, 'mymod')

[{'caller': 'mymod.wrapper_pos',
  'callee': 'mymod.target',
  'kind': 'delegates',
  'confidence': 0.85},
 {'caller': 'mymod.wrapper_kw',
  'callee': 'mymod.to=target',
  'kind': 'delegates',
  'confidence': 0.85}]

In [ ]:
_PATCH_SRC = '''
from fastcore.all import patch, delegates
from apswutils.db import Database, Table, View

@patch
def basic(self:Database, x): pass

@patch(as_prop=True)
def prop(self:Database): pass

@patch(cls_method=True)
def cls_m(cls:Database): pass

@patch
def no_ann(self): pass

@patch
def union_m(self:Table|View): pass
'''
dyn_edges(_PATCH_SRC, 'mymod')

[{'caller': 'mymod.Database',
  'callee': 'mymod.basic',
  'kind': 'patch',
  'confidence': 0.9},
 {'caller': 'mymod.Database',
  'callee': 'mymod.prop',
  'kind': 'patch',
  'confidence': 0.9},
 {'caller': 'mymod.Database',
  'callee': 'mymod.cls_m',
  'kind': 'patch',
  'confidence': 0.9},
 {'caller': 'mymod.Table',
  'callee': 'mymod.union_m',
  'kind': 'patch',
  'confidence': 0.9},
 {'caller': 'mymod.View',
  'callee': 'mymod.union_m',
  'kind': 'patch',
  'confidence': 0.9}]

In [ ]:
#| export
def _build_imap(sources=None, filenames=None, root=None):
	'Build {module: {local_name: qualified_name}} for resolving None-namespace callee nodes.'
	if filenames:
		fn = lambda p: '.'.join(Path(p).relative_to(root).with_suffix('').parts)
		srcs = {fn(p): Path(p).read_text(errors='replace') for p in filenames}
	else: srcs = sources or {}
	return L(set(srcs)).map_dict(lambda m: parse(srcs[m])[-1] or {})

def static_edges(sources:dict[str,str]=None, filenames:list[str]=None, root:str=None) -> tuple[list[dict], dict]:
	"Static call edges via pyan3. Provide either filenames+root or sources dict."
	assert bool(filenames) ^ bool(sources), 'Provide one of sources or filenames, but not both'
	if filenames and root is None: root = str(Path(filenames[0]).parent.parent)
	try:
		from pyan.analyzer import CallGraphVisitor
		if filenames: v = CallGraphVisitor(filenames, root=root)
		else: v = CallGraphVisitor.from_sources(tuplify((s,m) for m,s in sources.items()))
		v.postprocess()
		imap = _build_imap(sources=sources, filenames=filenames, root=root)
		def _callee(c, t):
			if t.namespace and t.namespace != '*': return f'{t.namespace}.{t.name}'
			if t.namespace is None:
				parts = c.namespace.split('.')
				for i in range(len(parts), 0, -1):
					if (m:='.'.join(parts[:i])) in imap and t.name in imap[m]: return imap[m][t.name]
			return None
		return ([_e(f'{c.namespace}.{c.name}', cl, 'static', 1.0)
				 for c,ts in v.uses_edges.items() if c.namespace and c.namespace != '*'
				 for t in ts if '^^^' not in (t.name or '') and (cl := _callee(c,t))],
				{f"{n.namespace}.{n.name}":
					 {'node':f"{n.namespace}.{n.name}",
					  'flavor':str(n.flavor).split('.')[-1].lower(), 'file':n.filename or ''}
				 for nlist in v.nodes.values() for n in nlist if hasattr(n,'name') and n.namespace})
	except Exception as ex: print(f"pyan3 static analysis failed: {ex}"); return [], {}

In [ ]:
edges, meta = static_edges({"mod": "def foo(): pass\ndef bar(): foo()"})
meta

{'mod.foo': {'node': 'mod.foo', 'flavor': 'function', 'file': 'mod'},
 'mod.foo.^^^argument^^^': {'node': 'mod.foo.^^^argument^^^',
  'flavor': 'unspecified',
  'file': 'mod'},
 'mod.bar.^^^argument^^^': {'node': 'mod.bar.^^^argument^^^',
  'flavor': 'unspecified',
  'file': 'mod'},
 'mod.bar': {'node': 'mod.bar', 'flavor': 'function', 'file': 'mod'}}

In [ ]:
assert all(k in edges[0] for k in ('caller','callee','kind','confidence'))
assert all(e['kind'] == 'static' for e in edges)

In [ ]:
static_edges({"bad": "def (: !!!"})

pyan3 static analysis failed: invalid syntax (bad, line 1)


([], {})

In [ ]:
from textwrap import dedent
_src1 = dedent('''
class Database:
	def query(self): pass
''')
_src2 = dedent('''
from mod1 import Database
def use_db():
    db = Database()
    db.query()
''')
_edges, _ = static_edges({"mod1": _src1, "mod2": _src2})
assert not any(e['callee'].startswith('None.') for e in _edges), \
	f"None namespace callees found: {[e['callee'] for e in _edges if e['callee'].startswith('None.')]}"
print('cross-module callee fix ok:', [e['callee'] for e in _edges])

cross-module callee fix ok: ['mod1.Database', 'mod1.Database.query']


In [ ]:
static_edges(filenames=['/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/litesearch/core.py'], root=Path('/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages'))

([{'caller': 'litesearch.core._dtype_suffix',
   'callee': 'litesearch.core._dtype_suffixes',
   'kind': 'static',
   'confidence': 1.0},
  {'caller': 'litesearch.core.query',
   'callee': 'fastcore.all.Iterable',
   'kind': 'static',
   'confidence': 1.0},
  {'caller': 'litesearch.core.query',
   'callee': 'apswutils.utils.cursor_row2dict',
   'kind': 'static',
   'confidence': 1.0},
  {'caller': 'litesearch.core.query',
   'callee': 'fastcore.all.Union',
   'kind': 'static',
   'confidence': 1.0},
  {'caller': 'litesearch.core.query',
   'callee': 'fastcore.all.Generator',
   'kind': 'static',
   'confidence': 1.0},
  {'caller': 'litesearch.core.query',
   'callee': 'fastcore.all.Optional',
   'kind': 'static',
   'confidence': 1.0},
  {'caller': 'litesearch.core.query',
   'callee': 'fastcore.all.patch',
   'kind': 'static',
   'confidence': 1.0},
  {'caller': 'litesearch.core.query',
   'callee': 'fastlite.Database',
   'kind': 'static',
   'confidence': 1.0},
  {'caller': 'litesea

In [ ]:
#| export
def _nodes(db):
    'All node connectors buy caller and callee columns. O(E).'
    return L(r['node'] for r in db.q('''SELECT DISTINCT caller node FROM graph_edges
                                     UNION SELECT DISTINCT callee FROM graph_edges'''))

def _pagerank(db, nodes:set=None, alpha=0.85, iters=50, tol=1e-6):
    'Iterative PageRank with precomputed adjacency — O(E + N·iters).'
    nodes = nodes or _nodes(db)
    if not nodes: return {}
    n = len(nodes)
    pr = {nd: 1/n for nd in nodes}
    out,in_e = defaultdict(int), defaultdict(list)
    for r in db.t.graph_edges(select='caller,callee'):
	    out[r['caller']] += 1
	    in_e[r['callee']].append(r['caller'])
    for _ in range(iters):
	    new = {nd:(1-alpha)/n + alpha*sum(pr.get(c, 0)/(out[c] or 1) for c in in_e[nd]) for nd in nodes}
	    if max(abs(new[nd]-pr[nd]) for nd in nodes) < tol: break
	    pr = new
    return pr

In [ ]:
#| export
@patch
def _centrality(self: CodeGraph, nodes:set=None):
	db = self.db
	pr = _pagerank(db, nodes)
	in_d  = {r['nd']:r['n'] for r in db.q('SELECT callee nd,COUNT(*) n FROM graph_edges GROUP BY callee')}
	out_d = {r['nd']:r['n'] for r in db.q('SELECT caller nd,COUNT(*) n FROM graph_edges GROUP BY caller')}
	db.t.graph_nodes.insert_all([{'node':nd, 'pagerank':round(pr.get(nd,0),5), 'in_degree':in_d.get(nd,0),
	                              'out_degree':out_d.get(nd,0)} for nd in set(pr)], upsert=True, pk='node')
	return self

@patch
def _add_dyn(self:CodeGraph, mod_map:dict):
	"Add dynamic edges; mod_map values are src strings or Path objects."
	existing = {(r['caller'],r['callee']) for r in self.db.t.graph_edges(select='caller,callee')}
	d_edges, groups, new_nodes = [], [], set()
	for mod, val in mod_map.items():
		src = Path(val).read_text(errors='replace') if isinstance(val, os.PathLike) else val
		for e in dyn_edges(src, mod):
			fr,t = e['caller'], e['callee']
			if e['kind'] == 'co_dispatch':
				for grp in groups:
					if {fr,t} & grp: grp |= {fr,t}; break
				else: groups.append({fr,t})
			elif (fr,t) not in existing:
				d_edges.append(e)
				existing.add((fr,t))
			new_nodes |= {fr,t}
	if d_edges: self.db.t.graph_edges.insert_all(d_edges)
	if groups:
		base = first(self.db.q('SELECT COALESCE(MAX(group_id)+1,0) n FROM co_dispatch'))['n']
		self.db.t.co_dispatch.insert_all([{'group_id':base+i,'node':nd} for i,grp in enumerate(groups) for nd in grp])
	return new_nodes

def _lambda_nodes(sources:dict[str,str]=None, filenames:list[str]=None, root:str=None) -> dict:
	'Extract module-level `name = lambda ...` nodes; same signature as static_edges.'
	if filenames:
		fn = lambda p: '.'.join(Path(p).relative_to(root).with_suffix('').parts)
		sources = {fn(p): (Path(p).read_text(errors='replace'), str(p)) for p in filenames}
	else: sources = {mod: (src, '') for mod, src in (sources or {}).items()}
	return {f"{mod}.{t.id}": {'node':f"{mod}.{t.id}", 'flavor':'function', 'file':file}
			for mod, (src, file) in sources.items()
			for tree, *_ in [parse(src)] if tree
			for n in tree.body
			if isinstance(n, ast.Assign) and isinstance(n.value, ast.Lambda)
			for t in n.targets if isinstance(t, ast.Name)}

@patch
def _add_static(self:CodeGraph, sources:dict[str,str]=None, filenames:list[str]=None, root:str=None):
	"Add static edges from sources dict. Uses pyan3 for full-corpus analysis."
	s_edges, meta = static_edges(sources=sources, filenames=filenames, root=root)
	meta |= _lambda_nodes(sources=sources, filenames=filenames, root=root)
	if s_edges: self.db.t.graph_edges.insert_all(s_edges)
	if meta: self.db.t.graph_nodes.insert_all(meta.values(), upsert=True, pk='node')
	return set(meta) | set(L(s_edges).attrgot('caller')) | set(L(s_edges).attrgot('callee'))

@patch
def from_sources(self:CodeGraph, sources:dict[str,str]):
	'Build from {module_name: source_str}.'
	return self._centrality(self._add_static(sources=sources) | self._add_dyn(sources))

In [ ]:
g_pr = CodeGraph(':memory:')
g_pr.db['graph_edges'].insert_all([
    {'caller':'A','callee':'B','kind':'static','confidence':1.0},
    {'caller':'B','callee':'C','kind':'static','confidence':1.0},
    {'caller':'A','callee':'C','kind':'static','confidence':1.0},
])
pr = _pagerank(g_pr.db)
print(pr)
assert 'C' in pr, "C should be in pagerank"
assert pr['C'] > pr['A'], f"C (most linked-to) should rank higher than A: {pr}"
print("pagerank ok:", pr)

{'A': 0.05000000000000001, 'B': 0.07125000000000001, 'C': 0.13181250000000003}
pagerank ok: {'A': 0.05000000000000001, 'B': 0.07125000000000001, 'C': 0.13181250000000003}


In [ ]:
CodeGraph(':memory:').from_sources({'mod': 'def foo(): pass\ndef bar(): foo()\n'}).db.t.graph_nodes()

[{'node': 'mod.foo',
  'flavor': 'function',
  'file': 'mod',
  'pagerank': 0.06938,
  'in_degree': 1,
  'out_degree': 0},
 {'node': 'mod.foo.^^^argument^^^',
  'flavor': 'unspecified',
  'file': 'mod',
  'pagerank': 0.0375,
  'in_degree': 0,
  'out_degree': 0},
 {'node': 'mod.bar.^^^argument^^^',
  'flavor': 'unspecified',
  'file': 'mod',
  'pagerank': 0.0375,
  'in_degree': 0,
  'out_degree': 0},
 {'node': 'mod.bar',
  'flavor': 'function',
  'file': 'mod',
  'pagerank': 0.0375,
  'in_degree': 0,
  'out_degree': 1}]

In [ ]:
#| export
@patch
def process_files(self:CodeGraph,
    files,     # list of .py file paths to analyse
    root=None  # package root; auto-detected via __init__.py walk if None
):
	'Build static + dynamic call-graph edges for the given source files and recompute centrality.'
	if root is None or not files: root = files[0] if files else '.'
	pkg_dir = Path(str(files[0])).parent
	while has_init(pkg_dir): pkg_dir = pkg_dir.parent
	root = str(pkg_dir)
	nodes = self._add_static(filenames=files, root=root)
	fn = lambda p: '.'.join(Path(p).relative_to(root).with_suffix('').parts)
	nodes |= self._add_dyn({fn(p): Path(p) for p in files})
	return self._centrality(nodes)

@patch
@fdelegates(globtastic)
def from_dir(self:CodeGraph, dir: str | Path, **kwargs) -> 'CodeGraph':
	'Build from .py files under path — pyan reads files directly.'
	files = dir2files(dir, exts=['.py'], func=os.path.join, **kwargs)
	return self.process_files(files, Path(dir).parent)

@patch
def from_file(self: CodeGraph, path:str|Path) -> 'CodeGraph':
	'Build from a single .py file. Module name derived from filename.'
	p = Path(path).resolve()
	pkg_root, parts = p.parent, [p.with_suffix('').name]
	while has_init(pkg_root):
		parts.insert(0, pkg_root.name)
		pkg_root = pkg_root.parent
	return self._centrality(self._add_static(filenames=[str(p)], root=pkg_root) | self._add_dyn({'.'.join(parts): p}))

@patch
@fdelegates(globtastic)
def from_pkg(self: CodeGraph, pkg: str, **kwargs) -> 'CodeGraph':
    'Build from an installed package. Uses pyan3 on the package source files.'
    s = spec(pkg)
    if not s: raise ValueError(f"{pkg!r} not installed")
    p = Path(s.origin).parent if s.origin else Path(s.submodule_search_locations[0])
    return self.from_dir(p, **kwargs)

In [ ]:
cg=CodeGraph(':memory:')

In [ ]:
cg.from_pkg('httpx').from_pkg('fastcore').from_pkg('litesearch').from_pkg('fastlite').from_pkg('apswutils')

<__main__.CodeGraph>

In [ ]:
cg.from_pkg('protobuf')

<__main__.CodeGraph>

In [ ]:
cg.gn(select='*', where='node like "%get_store%"')

[{'node': 'litesearch.core.get_store.str',
  'flavor': 'attribute',
  'file': '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/litesearch/core.py',
  'pagerank': 0.00023,
  'in_degree': 0,
  'out_degree': 0},
 {'node': 'litesearch.core.get_store.^^^argument^^^',
  'flavor': 'unspecified',
  'file': '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/litesearch/core.py',
  'pagerank': 0.00023,
  'in_degree': 0,
  'out_degree': 0},
 {'node': 'litesearch.core.get_store',
  'flavor': 'function',
  'file': '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/litesearch/core.py',
  'pagerank': 0.00029,
  'in_degree': 1,
  'out_degree': 3},
 {'node': 'litesearch.core.get_store.bool',
  'flavor': 'attribute',
  'file': '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/litesearch/core.py',
  'pagerank': 0.00023,
  'in_degree': 0,
  'out_degree': 0},
 {'node': 'litesearch.core.get_store.^^^argument^^^.t',

In [ ]:
cg.from_dir('../kosha')

<__main__.CodeGraph>

In [ ]:
cg.db.t.graph_nodes.count

16160

In [ ]:
cg.gn('node like "%emb_doc%"')

[{'node': 'kosha.core.emb_doc.str',
  'flavor': 'attribute',
  'file': '../kosha/core.py',
  'pagerank': 0.00011,
  'in_degree': 0,
  'out_degree': 0},
 {'node': 'kosha.core.emb_doc',
  'flavor': 'function',
  'file': '../kosha/core.py',
  'pagerank': 0.00016,
  'in_degree': 1,
  'out_degree': 1},
 {'node': 'kosha.core.emb_doc.^^^argument^^^',
  'flavor': 'unspecified',
  'file': '../kosha/core.py',
  'pagerank': 0.00011,
  'in_degree': 0,
  'out_degree': 0},
 {'node': 'kosha.core.emb_doc.lambda.0.^^^argument^^^',
  'flavor': 'unspecified',
  'file': '../kosha/core.py',
  'pagerank': 0.00011,
  'in_degree': 0,
  'out_degree': 0},
 {'node': 'kosha.core.emb_doc.lambda.0',
  'flavor': 'scope',
  'file': '../kosha/core.py',
  'pagerank': 0.00011,
  'in_degree': 0,
  'out_degree': 1}]

In [ ]:
#| export
@patch
def callers(self: CodeGraph,
    n: str,           # fully-qualified node name (e.g. 'mymod.foo')
    with_kind=False   # if True, return (caller, edge_kind) tuples instead of plain names
) -> L:
    'All nodes that call `n`, optionally with edge kind.'
    fn = lambda r: (r['caller'], r['kind']) if with_kind else r['caller']
    return L(self.db.t.graph_edges(select='caller,kind', where='callee=?', where_args=[n])).map(fn)

@patch
def callees(self: CodeGraph,
    n: str,           # fully-qualified node name
    with_kind=False   # if True, return (callee, edge_kind) tuples instead of plain names
) -> L:
    'All nodes called by `n`, optionally with edge kind.'
    fn = lambda r: (r['callee'], r['kind']) if with_kind else r['callee']
    return L(self.db.t.graph_edges(select='callee,kind', where='caller=?', where_args=[n])).map(fn)

@patch
def neighbors(self: CodeGraph,
    n: str,       # fully-qualified node name
    depth: int=1  # number of hops to expand
) -> L:
    'callers + callees up to depth hops.'
    seen, frontier = set(), {n}
    for _ in range(depth):
        nxt = set()
        for node in frontier: nxt |= set(self.callers(node)) | set(self.callees(node))
        frontier = nxt - seen - {n}
        seen |= frontier
    return L(seen)

@patch
def co_dispatched(self: CodeGraph,
    node: str  # fully-qualified node name
) -> L:
    'Nodes registered in the same dispatch group as `node` (route list, handler table, etc.).'
    if not (rows := self.db.t.co_dispatch(select='group_id', where='node=?', where_args=[node])): return L()
    o=self.db.t.co_dispatch(select='node', where='group_id=? and node!=?', where_args=[rows[0]['group_id'], node])
    return L(o).attrgot('node')

@patch
def ranked(self: CodeGraph,
    k: int=10,          # number of top nodes to return
    module: str=None    # optional module prefix filter (e.g. 'fastcore.basics')
) -> L:
    'Top-k nodes by pagerank, optionally filtered to a module prefix.'
    kw = dict(where='node LIKE ?', where_args=[f"{module}.%"]) if module else {}
    return L(self.db.t.graph_nodes(select='node,pagerank', order_by='pagerank DESC', limit=k, **kw))

@patch
def _bfs(self: CodeGraph, src:str, targets:set) -> dict:
	'BFS from src; returns {tgt: L(path)} for each reachable target.'
	par, fnd, rem = {src: None}, {}, set(targets) - {src}
	fro = [src]
	_get = lambda f: self.ge(select='caller,callee', where=f'caller IN ({",".join("?" * len(f))})', where_args=f)
	while fro and rem:
		nxt = []
		for r in dict2obj(_get(fro)):
			if r.callee in par: continue
			par[r.callee] = r.caller
			if r.callee in fnd: continue
			path, n = [], r.callee
			while n is not None: path.append(n); n = par[n]
			fnd[r.callee] = L(reversed(path))
			rem.discard(r.callee)
			nxt.append(r.callee)
		fro = nxt
	return fnd

@patch
def short_path(self: CodeGraph,
    src: str,  # fully-qualified source node name
    tgt: str   # fully-qualified target node name
) -> L:
    'Shortest path from src to tgt via BFS. Returns L of node names or empty L.'
    if src == tgt: return L([src])
    return self._bfs(src, {tgt}).get(tgt, L())

@patch
def short_paths(self: CodeGraph,
    nodes,      # iterable of fully-qualified node names
    k: int=10   # consider only the first k nodes
) -> L:
    'Shortest paths between all pairs in top-k nodes — one BFS per source node.'
    ns = listify(nodes)[:k]
    return L(ns).map(lambda s: self._bfs(s, set(ns)-{s}).values()).concat().filter(true).sorted(key=len, reverse=True)

@patch
def edge_kinds(self: CodeGraph) -> dict:
    'Count of each edge kind in the graph (static, decorator, patch, registration, etc.).'
    return {r['kind']: r['n'] for r in self.db.q("SELECT kind, COUNT(*) n FROM graph_edges GROUP BY kind")}

@patch
def node_info(self: CodeGraph,
    node: str,        # fully-qualified node name
    with_kind=False   # include edge kind in callers/callees tuples
) -> dict:
    'Full node record: graph_nodes row merged with callers, callees, and co-dispatch peers.'
    meta = first(self.gn(xtra=dict(node=node))) or {}
    edges = dict(callers=self.callers(node,with_kind), callees=self.callees(node,with_kind))
    return meta | edges | dict(co_dispatched=self.co_dispatched(node))

In [ ]:
cg.db.t.graph_edges(where='caller like "%apswutils.db.Table.insert_all%" and callee like "%insert_chunk%"')

[{'id': 9744,
  'caller': 'apswutils.db.Table.insert_all',
  'callee': 'apswutils.db.Table.insert_chunk',
  'kind': 'static',
  'confidence': 1.0}]

In [ ]:
to_date_co = cg.co_dispatched('fastcore.basics.to_date')
assert all('fastcore' in n for n in to_date_co), \
    f"fastcore.basics.to_date co_dispatched leaked into httpx: {to_date_co}"
assert len(to_date_co) > 0, "fastcore.basics.to_date should have co_dispatched peers"
print("co_dispatched cross-package isolation ok:", to_date_co)

co_dispatched cross-package isolation ok: ['fastcore.basics.to_int', 'fastcore.basics.to_bool', 'fastcore.basics.to_float']


In [ ]:
cg.node_info('fastcore.basics.to_date')

{'node': 'fastcore.basics.to_date',
 'flavor': 'function',
 'file': '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/basics.py',
 'pagerank': 3e-05,
 'in_degree': 0,
 'out_degree': 2,
 'callers': [],
 'callees': ['fastcore.basics._typeerr', 'fastcore.basics.str2date'],
 'co_dispatched': ['fastcore.basics.to_int', 'fastcore.basics.to_bool', 'fastcore.basics.to_float']}

In [ ]:
n = cg.db.t.graph_nodes(where='node like "%mk_write"')[0]['node']
cg.node_info(n)

{'node': 'fastcore.xtras.mk_write',
 'flavor': 'function',
 'file': '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/xtras.py',
 'pagerank': 4e-05,
 'in_degree': 2,
 'out_degree': 2,
 'callers': ['fastcore.xtras.write_json', 'fastcore.xtras.Path'],
 'callees': ['fastcore.imports.NoneType', 'fastcore.basics.patch'],
 'co_dispatched': []}

In [ ]:
path = cg.short_path('apswutils.db.Table.upsert', 'apswutils.db.Table.insert_chunk')
assert path[0] == 'apswutils.db.Table.upsert', f"wrong start: {path}"
assert path[-1] == 'apswutils.db.Table.insert_chunk', f"wrong end: {path}"
assert len(path) >= 2, f"path too short: {path}"
assert cg.short_path('apswutils.db.Table.upsert', 'apswutils.db.Table.upsert') == L(['apswutils.db.Table.upsert']), "src==tgt should return [src]"
assert cg.short_path('apswutils.db.Table.upsert', 'no.such.node') == L(), "missing tgt should return empty"
print("short_path ok:", path)

short_path ok: ['apswutils.db.Table.upsert', 'apswutils.db.Table.upsert_all', 'apswutils.db.Table.insert_all', 'apswutils.db.Table.insert_chunk']


In [ ]:
cg.ranked(5, module='litesearch')

[{'node': 'litesearch.data.kw', 'pagerank': 0.0011}, {'node': 'litesearch.utils.FastEncode.__init__', 'pagerank': 0.00075}, {'node': 'litesearch.utils.download_model', 'pagerank': 0.00072}, {'node': 'litesearch.data.file_parse.meta_', 'pagerank': 0.00064}, {'node': 'litesearch.utils.FastEncode._load_enc', 'pagerank': 0.00053}]

In [ ]:
cg.callers('litesearch.data.kw')

['litesearch.core.get_store', 'litesearch.core.database', 'litesearch.data.pkg2chunks', 'litesearch.data.pre', 'litesearch.utils.FastEncode.encode', 'litesearch.utils.FastEncode.encode.lambda.0', 'litesearch.utils.FastEncode.encode_document', 'litesearch.utils.FastEncode.encode_query', 'litesearch.utils.FastEncodeImage.embed', 'litesearch.utils.FastEncodeImage.embed.lambda.0', 'litesearch.utils.FastEncodeMultimodal.encode_text', 'litesearch.utils.FastEncodeMultimodal.encode_image', 'litesearch.utils.encode_pdf_texts']

In [ ]:
#| export
@patch
def _drop_file(self: CodeGraph, path: str):
    'Delete all graph_edges and graph_nodes belonging to this file path.'
    if not (nodes:=self.file2nodes(path)): return
    pl, nl = ','.join('?' * len(nodes)), list(nodes)
    self.db.q(f"DELETE FROM graph_edges WHERE caller IN ({pl}) OR callee IN ({pl})",nl*2)
    self.db.q(f"DELETE FROM graph_nodes WHERE node IN ({pl})", nl)
    self.db.q(f"DELETE FROM co_dispatch WHERE node IN ({pl})", nl)
    self.db.q("DELETE FROM file_index WHERE path=?", [path])

@patch
def file2nodes(self: CodeGraph, path: str) -> set:
    'Return set of node names whose file column matches path.'
    return set(L(self.db.t.graph_nodes(select='distinct node', where='file=?', where_args=[path])).attrgot('node'))

In [ ]:
g_drop = CodeGraph(':memory:')
g_drop.db.t.graph_nodes.insert_all([
    {'node':'mod.foo','flavor':'function','file':'/proj/mod.py'},
    {'node':'mod.bar','flavor':'function','file':'/proj/mod.py'},
])
g_drop.db.t.graph_edges.insert({'caller':'mod.foo','callee':'mod.bar','kind':'static','confidence':1.0})
assert g_drop.file2nodes('/proj/mod.py') == {'mod.foo','mod.bar'}
g_drop._drop_file('/proj/mod.py')
assert g_drop.file2nodes('/proj/mod.py') == set()
assert not list(g_drop.db.execute("SELECT * FROM graph_edges"))
print("_drop_file + _nodes_for_file ok")

_drop_file + _nodes_for_file ok


In [ ]:
#| export
@patch
def _index_files(self: CodeGraph, files, sc='repo'):
	'Add or update file_index entries for these files. root is for grouping, e.g. by dir or pkg.'
	o = [dict(path=str(f), root=sc, last_analyzed_at=Path(f).stat().st_mtime) for f in files]
	self.db.t.file_index.insert_all(o, replace=True)

@patch
def _stale(self: CodeGraph, files, sc=None) -> list:
	kw = dict(where='root=?', where_args=[sc]) if sc else {}
	known = {r['path']: r['last_analyzed_at'] for r in self.db.t.file_index(**kw)}
	return [f for f in files if Path(f).exists() and known.get(str(f)) != Path(f).stat().st_mtime]

In [ ]:
#| export
@patch
def sync_dir(self: CodeGraph, dir: str | Path, force=False) -> 'CodeGraph':
	'Incremental sync for a directory: drop missing files, update changed files, add new files.'
	if not dir: print('no action. dir empty'); return self
	files = dir2files(dir, exts=['.py'], func=os.path.join)
	known=set(L(self.db.t.file_index(select='distinct path', where='root=?', where_args=[str(dir)])).attrgot('path'))
	for f in known - set(files): self._drop_file(f)
	fs = files if force else self._stale(files, str(dir))
	if fs: self.process_files(fs, str(dir)) and self._index_files(fs, str(dir))
	return self

@patch
def sync_pkgs(self: CodeGraph, pkgs: list[str]) -> 'CodeGraph':
	'Incremental sync for packages: drop missing files, update changed files, add new files.'
	if not pkgs: print('no action. pkgs empty'); return self

	arun(parallel_async(self.from_pkg, pkgs))
	return self

@patch
def sync(self: CodeGraph, dir=None, pkgs=None, force=False) -> 'CodeGraph':
	'Incremental sync: from_dir per dir, from_pkg per pkg; manage file_index.'
	self.sync_dir(dir, force) and self.sync_pkgs(pkgs)
	return self

In [ ]:
import tempfile, time

with tempfile.TemporaryDirectory() as tmp:
	pa, pb = Path(f'{tmp}/a.py'), Path(f'{tmp}/b.py')
	pa.write_text('def foo(): pass\n'); pb.write_text('def bar(): pass\n')

	g = CodeGraph(':memory:')
	g.sync(dir=tmp)
	fi = {r['path']: r['last_analyzed_at'] for r in g.db.t.file_index.rows}
	assert str(pa) in fi and str(pb) in fi, fi

	# Modify b.py — mtime changes
	time.sleep(0.01)
	pb.write_text('def bar(): return 42\n')
	pb.touch()  # ensure mtime update
	g.sync(dir=tmp)
	fi2 = {r['path']: r['last_analyzed_at'] for r in g.db.t.file_index.rows}
	assert fi2[str(pa)] == fi[str(pa)], f"a.py should not be reprocessed: {fi2}"
	assert fi2[str(pb)] != fi[str(pb)], f"b.py should be reprocessed: {fi2}"

	# Delete a.py — should be removed from file_index
	pa.unlink()
	g.sync(dir=tmp)
	fi3 = {r['path'] for r in g.db.t.file_index.rows}
	assert str(pa) not in fi3, f"a.py should be removed: {fi3}"
	print("sync incremental ok, remaining files:", fi3)

	# add packages to codegraph
	g.sync(pkgs=['httpx', 'fastcore'])
	assert len(g.gn(where='node like "httpx.%"')) > 0, "httpx nodes should be in graph"
	assert len(g.gn(where='node like "fastcore.%"')) > 0, "fastcore nodes should be in graph"


no action. pkgs empty
no action. pkgs empty
no action. pkgs empty
sync incremental ok, remaining files: {'/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/tmpg_9d9f_w/b.py'}
no action. dir empty


In [ ]:
#| export
@patch(as_prop=True)
def graph(self: Kosha) -> CodeGraph:
	'code graph for this kosha instance'
	return CodeGraph(self.root.joinpath('.kosha','graph.db'))

@patch(as_prop=True)
def graphdb(self: Kosha):
	'Underlying graph.db database for direct queries.'
	return self.graph.db

@patch(as_prop=True)
def gn(self:Kosha):
	'graph_nodes table for direct queries.'
	return self.graph.db.t.graph_nodes

@patch(as_prop=True)
def ge(self:Kosha):
	'graph_edges table for direct queries.'
	return self.graph.db.t.graph_edges

@patch(as_prop=True)
def ni(self:Kosha):
	'Node info helper: graph_nodes row + callers, callees, co-dispatched nodes.'
	return self.graph.node_info

@patch(as_prop=True)
def neighbors(self:Kosha):
	'Neighbors helper: callers + callees up to depth hops.'
	return self.graph.neighbors

@patch(as_prop=True)
def short_path(self:Kosha):
	'Shortest path helper: shortest path between two nodes.'
	return self.graph.short_path

@patch(as_prop=True)
def short_paths(self:Kosha):
	'Shortest paths helper: shortest paths between all pairs in top-k nodes.'
	return self.graph.short_paths

In [ ]:
#| export
@patch
def public_api(self: Kosha,
               pkg: str,         # package or module to index (e.g. 'kosha', 'litesearch.data')
               meta_cols:str='mod_name,docstring', # metadata keys to extract
               limit: int = 200  # max names to return
               ) -> L:
	'Return public API: all public_api=1 entries + @patch-derived methods from the call graph from env and repo stores.'
	fn = lambda r: r | dict(metadata=jl(r['metadata'])) if isinstance(r.get('metadata'), str) else r
	_get= lambda r, k: r.get('metadata', {}).get(k)
	je = lambda o,e='=',v='': 'json_extract(metadata,"$.%s") %s %s'%(o,e,v)
	p = pkg.split('.')[0]
	pkg_ = Path(s.origin).parent.name if (s:=spec(p)) else p.replace('-', '_')
	patch_ = self.ge(select='callee',where="kind='patch' AND caller LIKE ?",where_args=[f'{pkg}.%'])
	wh, kw = je('public_api',v='1'), dict(where_args=None)
	if patch_:
		_ph = '(%s)' % ','.join('?' * len(patch_))
		wh += ' OR ' + je('mod_name','in',_ph)
		kw = dict(where_args=L(patch_).attrgot('callee'))
	_dir_cond = je('dir','like',repr(pkg_))
	er = self.env_st(where=f"package={pkg_!r} AND ({wh})",limit=limit,**kw)
	rr = self.code_st(where=f'{_dir_cond} AND {wh}',limit=limit,**kw)
	return L(er+rr).map(fn).map(lambda r: {m: _get(r,m) for m in meta_cols.split(',')})[:limit]


In [ ]:
k=Kosha()
api = k.public_api('kosha')
names = {r['mod_name'] for r in api}
# sync, context etc. are @patch-ed onto Kosha — must appear now
assert any('sync' in n for n in names), f"patched 'sync' missing from public_api: {names}"
api_httpx = k.public_api('httpx')
assert len(api_httpx) > 0, 'httpx __all__-based entries should still be returned'
print('public_api patch-aware ok | kosha:', len(api), '| httpx:', len(api_httpx))
print('kosha sample:', sorted(names)[:8])

public_api patch-aware ok | kosha: 113 | httpx: 200
kosha sample: ['kosha.core._is_pkg_ingested', 'kosha.core.awatch_repo', 'kosha.core.env_context', 'kosha.core.nuke', 'kosha.core.pkg_context', 'kosha.core.pkgs2consider', 'kosha.core.pkgs_in_env', 'kosha.core.process_env']


In [ ]:
k.public_api('litesearch.data')[:3]

[{'mod_name': 'litesearch.data.pdf_texts', 'docstring': None}, {'mod_name': 'litesearch.data.pdf_links', 'docstring': None}, {'mod_name': 'litesearch.data.pdf_images', 'docstring': None}]

In [ ]:
k.public_api('fastlite', meta_cols='mod_name')

[{'mod_name': 'fastlite.core.t'}, {'mod_name': 'fastlite.core.c'}, {'mod_name': 'fastlite.core.c'}, {'mod_name': 'fastlite.core.__str__'}, {'mod_name': 'fastlite.core.__str__'}, {'mod_name': 'fastlite.core.q'}, {'mod_name': 'fastlite.core.link_dcs'}, {'mod_name': 'fastlite.core.__call__'}, {'mod_name': 'fastlite.core.selectone'}, {'mod_name': 'fastlite.core.set_classes'}, {'mod_name': 'fastlite.core.get_tables'}, {'mod_name': 'fastlite.core.v'}, {'mod_name': 'fastlite.core.create'}, {'mod_name': 'fastlite.core.import_file'}, {'mod_name': 'fastlite.kw.xtra'}, {'mod_name': 'fastlite.kw.get_last'}, {'mod_name': 'fastlite.kw.ids_and_rows_where'}, {'mod_name': 'fastlite.kw.get'}, {'mod_name': 'fastlite.kw.__getitem__'}, {'mod_name': 'fastlite.kw.create'}, {'mod_name': 'fastlite.kw.transform'}, {'mod_name': 'fastlite.kw.transform_sql'}, {'mod_name': 'fastlite.kw.update'}, {'mod_name': 'fastlite.kw.insert_all'}, {'mod_name': 'fastlite.kw.insert'}, {'mod_name': 'fastlite.kw.upsert'}, {'mod_nam

In [ ]:
k.ni('fastlite.kw.database')

{'node': 'fastlite.kw.database',
 'flavor': 'function',
 'file': '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastlite/kw.py',
 'pagerank': 0.00042,
 'in_degree': 0,
 'out_degree': 12,
 'callers': [],
 'callees': ['apsw.Connection', 'apswutils.db.Database', 'typing.Any', 'apswutils.db.Database', 'typing.Any', 'apsw.Connection', 'typing.Any', 'apswutils.db.Database', 'apsw.Connection', 'apsw.Connection', 'apswutils.db.Database', 'typing.Any'],
 'co_dispatched': []}

In [ ]:
#| export
@patch
def sync(self: Kosha,
         pkgs=None, # list of package names to sync (e.g. ['httpx', 'fastcore']); if None, sync all env packages
         dir=None, # directory to sync; if None, sync all of root
         verbose=True, # print progress messages
         in_parallel=False, # run repo, env, and graph sync in parallel
         force=False, # ignore file mtimes and reprocess all files
         pyproject=True, # if True, auto-detect env packages from pyproject.toml; if False, use pkgs argument as-is
         depth=1, # depth for pyproject env package detection; ignored if pyproject=False
         embed=True # whether to embed
 ) -> 'Kosha':
	'Sync code store, env store, and code graph. Runs in a daemon thread by default.'
	dir, pkgs = dir or self.root, pkgs or set(env_pkg_versions(pyproject,depth))
	g_pkgs = set(pkgs) if force else set(pkgs) - set(L(self.pkgs(select='distinct name')).attrgot('name'))
	ts = [bind(self.update_repo, dir, verbose=verbose, force=force, embed=embed),
		  bind(self.update_pkgs, pkgs, verbose=verbose, force=force, embed=embed),
		  bind(self.graph.sync, dir=dir, pkgs=g_pkgs, force=force)]
	if in_parallel: return arun(parallel_async(lambda f: f(), ts))
	else: return L(ts).map(lambda f: f())

@patch
def context(self: Kosha,
			q: str,               # query with optional key:value filters
			limit: int = 50,
			repo: bool = True,
			env: bool = True,
			graph: bool = True,
			compact: bool = False, # return slim dicts (mod_name, signature, docstring, lineno) instead of full chunks
            columns:str='content,metadata',
            sys_wide=True,
			**kw                  # forwarded to env_context / repo_context
) -> L:
	'Fan-out semantic search: parse filters, run repo + env searches, merge with chained RRF.'
	def _tag(res, pref): return L(res).map(lambda r: r | dict(_src_id=f'{pref}:{r.get("rowid", id(r))}'))
	items = []
	if repo: items.append(('repo', bind(self.repo_context, q, limit=limit*2, columns=columns, **kw)))
	if env: items.append(('env', bind(self.env_context, q, limit=limit*2, columns=columns,sys_wide=sys_wide, **kw)))
	results = L(arun(parallel_async(lambda f: (f[0],f[1]()), items)))
	results = results.map(lambda r: _tag(r[1], r[0]))
	rrf = bind(rrf_merge, limit=limit*2, id_key='_src_id')
	res = L(L(results[1:]).reduce(lambda m,rs: rrf(m,rs), results[0]))
	if compact:
		meta = lambda r: filter_keys(r['metadata'],in_(['mod_name','docstring','lineno','path']))
		return res.map(lambda r:meta(r)|dict(sig=(r['content'].splitlines()[0])))[:limit]
	if not graph: return res[:limit]
	return dict2obj(res.map(lambda r: r | self.ni(r['metadata']['mod_name'])))[:limit]

In [ ]:
k = Kosha()

In [ ]:
# k.sync(L(k.pkgs(select='distinct name')).attrgot('name'), in_parallel=True)

In [ ]:
k.gn(select='*', where='node like "%get_store"')

[{'node': 'litesearch.core.get_store',
  'flavor': None,
  'file': None,
  'pagerank': 5e-05,
  'in_degree': 1,
  'out_degree': 0}]

In [ ]:
# k.sync(in_parallel=True, depth=2)

In [ ]:
k.ni('fastcore.basics.merge')

{'node': 'fastcore.basics.merge',
 'flavor': 'function',
 'file': '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/basics.py',
 'pagerank': 3e-05,
 'in_degree': 1,
 'out_degree': 10,
 'callers': ['fastcore.script.call_parse._f'],
 'callees': ['fastcore.basics.NS.__iter__', 'fastcore.nbio.Notebook.__iter__', 'fastcore.xtras.SaveReturn.__iter__', 'fastcore.xml.FT.__iter__', 'fastcore.docscrape.NumpyDocString.__iter__', 'fastcore.xtras._save_iter.__iter__', 'fastcore.xtras.IterLen.__iter__', 'fastcore.foundation.CollBase.__iter__', 'fastcore.xtras.CachedIter.__iter__', 'fastcore.basics.NotStr.__iter__', 'fastcore.xtras.IterLen.__iter__', 'fastcore.basics.NS.__iter__', 'fastcore.foundation.CollBase.__iter__', 'fastcore.xml.FT.__iter__', 'fastcore.xtras._save_iter.__iter__', 'fastcore.docscrape.NumpyDocString.__iter__', 'fastcore.nbio.Notebook.__iter__', 'fastcore.xtras.SaveReturn.__iter__', 'fastcore.xtras.CachedIter.__iter__', 'fastcore.basics.NotStr.__it

In [ ]:
first(k.context('how do I merge dicts package:fastcore path:basics', limit=1, compact=True))

{'path': '/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastcore/basics.py',
 'lineno': 655,
 'mod_name': 'fastcore.basics.merge',
 'docstring': 'Merge all dictionaries in `ds`',
 'sig': 'def merge(*ds):'}

In [ ]:
#| export
def is_sys_pkg(pkg:str) -> bool:
	'Return True if pkg is a system-wide package (e.g. in site-packages), False if likely user code.'
	from importlib.metadata import distribution as dist, PackageNotFoundError
	try: return bool(dist(pkg))
	except PackageNotFoundError: return False

@patch
def dep_stack(self:Kosha, seeds:list=None, depth:int=1, no_sys_pkg=True, allow:list=None) -> list:
	'BFS over pkg_deps, ordered by coupling strength.'
	seen, frontier, layers = set(seeds), set(seeds), [list(seeds)]
	fn=lambda p: p['tgt'] not in seen and (not no_sys_pkg or is_sys_pkg(p['tgt'])) and (not allow or p['tgt'] in allow)
	for _ in range(depth):
		if not frontier: break
		_pl = ','.join('?'*len(frontier))
		kw = dict(where=f'from_pkg IN ({_pl})', where_args=list(frontier))
		nxt = set(L(self.env_pd(select='to_pkg tgt,n_modules n',**kw)).filter(fn).attrgot('tgt'))
		if not nxt: break
		layers+=[nxt]
		seen |= nxt
		frontier = nxt
	return layers

@patch
def top_nodes(self:Kosha, pkg:str, k:int=5) -> L:
	'Top-k public API nodes for pkg ranked by pagerank in the code graph.'
	mods = self.public_api(pkg, meta_cols='mod_name').attrgot('mod_name')
	if not mods: return L()
	pl = ','.join('?'*len(mods))
	o = self.gn(select='node,pagerank', order_by='pagerank DESC', limit=k, where=f'node IN ({pl})', where_args=mods)
	return L(o).attrgot('node')

@patch
def api_call_paths(self:Kosha,
	from_pkg: str,  # package whose public API is the call source
	to_pkg: str,    # package whose public API is the call target
	k: int = 15     # top-k API nodes per package to consider
) -> dict:
	'Shortest call-graph paths from from_pkg public API nodes to to_pkg public API nodes.'
	f,a = self.top_nodes(from_pkg, k), self.top_nodes(to_pkg, k)
	paths = {}
	for fp in f: paths = paths | self.graph._bfs(fp, set(a))
	return filter_keys(paths, in_(a))


In [ ]:
rows = list(k.envdb.t.pkg_deps(where='from_pkg="fastlite"'))
assert rows, 'pkg_deps should have rows after update_pkg'
assert any(r['to_pkg'] in ('fastcore','fastlite') for r in rows), f'expected fastcore/fastlite: {rows[:3]}'
print('pkg_deps via update_pkg ok:', rows[:3])

pkg_deps via update_pkg ok: [{'from_pkg': 'fastlite', 'to_pkg': 'apsw', 'n_modules': 1}, {'from_pkg': 'fastlite', 'to_pkg': 'apswutils', 'n_modules': 2}, {'from_pkg': 'fastlite', 'to_pkg': 'dataclasses', 'n_modules': 2}]


In [ ]:
layers = k.dep_stack(['fastlite'])
print(layers)
print('-------------------------------------------------')
stack = L(layers).flatten()
assert 'fastcore' in stack, f'fastcore should be in dep_stack: {stack}'
print('_dep_stack ok:', stack)

[['fastlite'], {'fastcore', 'apswutils', 'apsw'}]
-------------------------------------------------
_dep_stack ok: ['fastlite', 'fastcore', 'apswutils', 'apsw']


In [ ]:
k.dep_stack(['fastlite'], allow=['fastcore'])

[['fastlite'], {'fastcore'}]

In [ ]:
k.api_call_paths('fastlite', 'apswutils')

{'apswutils.db.Table': ['fastlite.kw.ids_and_rows_where', 'apswutils.db.Table'],
 'apswutils.utils.rows_from_file': ['fastlite.core.import_file', 'apswutils.utils.rows_from_file'],
 'apswutils.utils.TypeTracker': ['fastlite.core.import_file', 'apswutils.utils.TypeTracker']}

In [ ]:
k.context('payments page monsterui fasthtml', repo=False)[:10]

[{'rowid': 79857, 'content': 'def account_management(sess):\n    user_email = sess[\'auth\']\n    user_payments = payments("email=?", (user_email,))\n    # Create table rows for each payment\n    payment_rows = []\n    for payment in user_payments:\n        action_button = ""\n        if payment.payment_status == \'paid\':\n            action_button = Button("Request Refund", hx_post=f"/refund?checkout_sid={payment.checkout_session_id}",hx_target="#refund-status")\n        elif payment.payment_status == \'refunded\': action_button = "Refunded"\n        \n        # Add row to table\n        payment_rows.append(\n            Tr(*map(Td, (payment.created_at, payment.amount, payment.payment_status, action_button))))\n    \n    # Create payment history table\n    payment_history = Table(\n        Thead(Tr(*map(Th, ("Date", "Amount", "Status", "Action")))),\n        Tbody(*payment_rows))\n    \n    return Titled(\n        "Account Management",\n        Div(H2(f"Account: {user_email}"),\n    

In [ ]:
k.context('payments page packages:monsterui,python-fasthtml', repo=False)[:10]

[{'rowid': 49813, 'content': 'def HTMX(path="/", host=\'localhost\', app=None, port=8000, height="auto", link=False, iframe=True, client=None):\n    "An iframe which displays the HTMX application in a notebook."\n    if isinstance(height, int): height = f"{height}px"\n    scr = """{\n        let frame = this;\n        window.addEventListener(\'message\', function(e) {\n            if (e.source !== frame.contentWindow) return; // Only proceed if the message is from this iframe\n            if (e.data.height) frame.style.height = (e.data.height+1) + \'px\';\n        }, false);\n    }""" if height == "auto" else ""\n    proto = \'http\' if host==\'localhost\' else \'https\'\n    fullpath = f"{proto}://{host}:{port}{path}" if host else path\n    src = f\'src="{fullpath}"\'\n    if link: display(HTML(f\'<a href="{fullpath}" target="_blank">Open in new tab</a>\'))\n    if isinstance(path, (FT,tuple,Safe)):\n        assert app, \'Need an app to render a component\'\n        route = f\'/{unqid

In [ ]:
#| export
@patch
def status(self: Kosha) -> dict:
	'Index freshness: file/pkg/node counts and stale file count.'
	known = {r['path']: r['uploaded_at'] for r in self.code_st(select='path,uploaded_at') if r['path']}
	stale = sum(1 for p, mt in known.items() if Path(p).exists() and Path(p).stat().st_mtime != mt)
	pkgs=merge(*L(self.pkgs(select='name,version')).map(lambda r: {r['name']: r['version']}))
	e = env_pkg_versions(pyproject=False)
	pkgs = filter_keys(pkgs, in_(e))
	sp = [k for k in pkgs if pkgs[k] != e[k]]
	return dict(files=len(known), packages=self.pkgs.count, graph_nodes=self.gn.count, stale_files=stale, stale_pkgs=sp)

In [ ]:
s = k.status()
assert set(s) == {'files', 'packages', 'graph_nodes', 'stale_files', 'stale_pkgs'}, f"unexpected status keys: {s}"
print('status ok:', s)

status ok: {'files': 3, 'packages': 87, 'graph_nodes': 78522, 'stale_files': 0, 'stale_pkgs': ['jupyter_server', 'jupyterlab', 'model2vec', 'huggingface_hub', 'beautifulsoup4']}


In [ ]:
#| export
@patch
def where_to_add(self: Kosha, description: str, limit: int = 5) -> L:
	'Likely insertion points for new code: file + line from top context results + co_dispatched peers.'
	results = self.context(description, graph=True, limit=limit)
	je = lambda v: f"json_extract(metadata,'$.mod_name')={v!r}"
	fn = lambda r: r | dict(metadata=jl(r['metadata'])) if 'metadata' in r else r
	locfn = lambda m: dict(path=m.get('path'),lineno=m.get('lineno'),end_lineno=m.get('end_lineno') or m.get('lineno'))
	def _loc(n):
		r = L(self.code_st(select='metadata', where=je(n), limit=1)).map(fn)
		return dict(node=n) | locfn(r[0]['metadata']) if r else {}
	out, seen = [], set()
	for r in results:
		mod = r['metadata'].get('mod_name', '')
		if not mod or mod in seen: continue
		seen.add(mod)
		loc = _loc(mod)
		if not loc or not loc['path']: continue
		co = list(r.get('co_dispatched', []))
		peer_ends = [p['end_lineno'] for c in co if (p := _loc(c)) and p['path'] == loc['path'] and p['end_lineno']]
		insert_after = max([loc['end_lineno'] or 0] + peer_ends, default=loc['lineno'])
		out+=[dict(node=mod,path=loc['path'],insert_after=insert_after,co_dispatched=co,pagerank=r.get('pagerank',0))]
	return L(out)

In [ ]:
pts = k.where_to_add('add a new CLI command', limit=3)
assert isinstance(pts, L)
if pts:
    assert all(k in pts[0] for k in ('node','path','insert_after','co_dispatched','pagerank'))
    print('where_to_add ok:', pts[0]['path'], 'line', pts[0]['insert_after'])
else: print('where_to_add ok: no results (index may be empty)')

where_to_add ok: /Users/71293/code/personal/orgs/kosha/kosha/cli.py line 247


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()

/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastprogress/fastprogress.py:171: UserWarning: Couldn't import ipython display functions, progress bar will use console behavior
  warn("Couldn't import ipython display functions, progress bar will use console behavior")
